In [1]:
# === Cell 1: Imports and data loading ===
import pandas as pd
import numpy as np
import plotly.graph_objects as go

disease_associations = pd.read_parquet('data/disease_associations.parquet')
cpm = pd.read_parquet('data/sample_by_gene.parquet')
pca_summary = pd.read_csv('data/pca_summary.csv').set_index('sample_type')

print(f"Loaded {len(disease_associations)} disease associations")
print(f"Loaded CPM matrix: {cpm.shape}")
print(f"Loaded {len(pca_summary)} treatments from pca_summary")

Loaded 705265 disease associations
Loaded CPM matrix: (48, 78987)
Loaded 21 treatments from pca_summary


The convention picks "2-fold" as a rough threshold for "biologically meaningful change" because:

Below 2-fold changes are often within the noise band of expression measurement technology (microarrays were noisy, RNA-seq is less so but still has variability)
Two-fold is intuitive — you can verbalize it ("expression doubled") and biologists tend to think in fold units
Effects smaller than 2-fold often don't translate to detectable phenotypic changes downstream

In [2]:
# === Cell 2: Identify Pareto frontier and compute per-gene log2FC ===
LOG2FC_THRESHOLD = 1.0
pseudocount = 0.5

# Pareto frontier already identified upstream — just filter
pareto_treatments = pca_summary[pca_summary['pareto_optimal']].index.tolist()
print("Pareto frontier treatments:", pareto_treatments)

# Per-gene log2FC per frontier treatment
gene_cols = [c for c in cpm.columns if c.startswith('ENSG')]
control_mask = cpm['sample_type'] == 'Control'
control_means = cpm.loc[control_mask, gene_cols].mean(axis=0)

off_target_results = {}
for tx in pareto_treatments:
    tx_mask = cpm['sample_type'] == tx
    tx_means = cpm.loc[tx_mask, gene_cols].mean(axis=0)
    log2fc = np.log2((tx_means + pseudocount) / (control_means + pseudocount))
    
    gene_log2fc = pd.DataFrame({
        'gene_id': gene_cols,
        'log2fc': log2fc.values,
        'abs_log2fc': np.abs(log2fc.values),
    })
    
    meaningful = gene_log2fc[gene_log2fc['abs_log2fc'] >= LOG2FC_THRESHOLD]
    off_target_results[tx] = meaningful
    print(f"{tx}: {len(meaningful)} genes with |log2FC| >= {LOG2FC_THRESHOLD}")

Pareto frontier treatments: ['hATF567', 'hATF561', 'nZF105', 'nZF139']
hATF567: 62 genes with |log2FC| >= 1.0
hATF561: 25 genes with |log2FC| >= 1.0
nZF105: 61 genes with |log2FC| >= 1.0
nZF139: 64 genes with |log2FC| >= 1.0


In [3]:
# === Cell 3: Per-treatment therapeutic area summary ===
for tx, top_genes in off_target_results.items():
    enriched = top_genes.merge(disease_associations, on='gene_id', how='left').dropna(subset=['disease_name'])
    exploded = enriched.explode('therapeutic_areas').dropna(subset=['therapeutic_areas'])
    
    area_summary = (exploded
                    .groupby('therapeutic_areas')
                    .agg(n_genes=('gene_id', 'nunique'),
                         n_diseases=('disease_name', 'nunique'),
                         avg_score=('association_score', 'mean'))
                    .sort_values(['n_genes', 'avg_score'], ascending=False))
    
    print(f"\n{'='*60}")
    print(f"TREATMENT: {tx}")
    print(f"  log2FC of SNHG14: {pca_summary.loc[tx, 'log2FC']:.3f}")
    print(f"  Distance to Control: {pca_summary.loc[tx, 'mean_dist']:.1f}")
    print(f"  Total meaningfully disturbed genes: {len(top_genes)}")
    print(f"  Therapeutic areas touched: {len(area_summary)}")
    print(f"{'='*60}")
    print(area_summary)


TREATMENT: hATF567
  log2FC of SNHG14: -0.501
  Distance to Control: 220.3
  Total meaningfully disturbed genes: 62
  Therapeutic areas touched: 24
                                              n_genes  n_diseases  avg_score
therapeutic_areas                                                           
measurement                                        17         282   0.289014
phenotype                                          13          62   0.314555
cancer or benign tumor                             13          41   0.191419
nervous system disease                             12          22   0.247576
musculoskeletal or connective tissue disease       11          27   0.303995
reproductive system or breast disease              10          17   0.246629
immune system disease                               9          10   0.239303
nutritional or metabolic disease                    8          14   0.372909
endocrine system disease                            8          15   0.287807
gast

In [4]:
# === Cell 4: Therapeutic area heatmap ===
area_rows = []
for tx in pareto_treatments:
    top_genes = off_target_results[tx]
    enriched = top_genes.merge(disease_associations, on='gene_id', how='left').dropna(subset=['disease_name'])
    exploded = enriched.explode('therapeutic_areas').dropna(subset=['therapeutic_areas'])
    
    area_counts = (exploded
                   .groupby('therapeutic_areas')
                   .agg(n_genes=('gene_id', 'nunique'))
                   .reset_index())
    area_counts['treatment'] = tx
    area_rows.append(area_counts)

area_long = pd.concat(area_rows, ignore_index=True)

# Normalize by total disturbed genes per treatment
treatment_totals = {tx: len(off_target_results[tx]) for tx in pareto_treatments}
area_long['pct_of_disturbed'] = area_long.apply(
    lambda r: r['n_genes'] / treatment_totals[r['treatment']] * 100, axis=1
)

# Pivot both views
count_data = area_long.pivot_table(
    index='therapeutic_areas', columns='treatment', values='n_genes', fill_value=0
)
pct_data = area_long.pivot_table(
    index='therapeutic_areas', columns='treatment', values='pct_of_disturbed', fill_value=0
)

# Sort by total impact
order = count_data.sum(axis=1).sort_values(ascending=True).index
count_data = count_data.loc[order]
pct_data = pct_data.loc[order]

# Build figure with toggleable views
fig = go.Figure()
fig.add_trace(go.Heatmap(
    z=count_data.values, x=count_data.columns, y=count_data.index,
    colorscale='Purples',
    text=count_data.values, texttemplate='%{text}', textfont=dict(size=11),
    colorbar=dict(title='# disturbed<br>genes linked'),
    visible=True,
    hovertemplate='Treatment: %{x}<br>Area: %{y}<br>Genes: %{z}<extra></extra>',
))
fig.add_trace(go.Heatmap(
    z=pct_data.values, x=pct_data.columns, y=pct_data.index,
    colorscale='Purples',
    text=pct_data.values, texttemplate='%{text:.1f}%', textfont=dict(size=11),
    colorbar=dict(title='% of disturbed<br>genes linked'),
    visible=False,
    hovertemplate='Treatment: %{x}<br>Area: %{y}<br>Share: %{z:.1f}%<extra></extra>',
))
fig.update_layout(
    updatemenus=[dict(
        type='buttons', direction='right',
        x=0.5, xanchor='center', y=1.12, yanchor='top',
        showactive=True,
        buttons=[
            dict(label='Raw counts', method='update',
                 args=[{'visible': [True, False]},
                       {'title': 'Therapeutic Area Disruption'}]),
            dict(label='Normalized %', method='update',
                 args=[{'visible': [False, True]},
                       {'title': 'Therapeutic Area Disruption'}]),
        ],
    )],
    title='Therapeutic Area Disruption',
    xaxis_title='Treatment', yaxis_title='',
    height=700, width=950,
    template='plotly_white',
)
fig.show()

In [5]:
# === Cell 5: Explore what's inside each top-level category ===
for area in ['disease', 'phenotype', 'biological process', 'measurement']:
    diseases_in_area = (disease_associations
        .explode('therapeutic_areas')
        .query("therapeutic_areas == @area")
        ['disease_name']
        .unique())
    
    print(f"\n{'='*60}")
    print(f"{area.upper()} — {len(diseases_in_area)} unique entries")
    print(f"{'='*60}")
    print(diseases_in_area[:30])


DISEASE — 0 unique entries
[]

PHENOTYPE — 4651 unique entries
['dermatomyositis' 'diabetes mellitus' 'diffuse scleroderma'
 'endocarditis' 'hypertension' 'infectious meningitis' 'preeclampsia'
 'obesity' 'morbid obesity' 'cellulitis' 'peripheral neuropathy' 'autism'
 'sign or symptom' 'psoriatic arthritis' 'sleep apnea'
 'left ventricular hypertrophy' 'ankylosing spondylitis'
 'deep vein thrombosis' 'hair color' 'chorea' 'androgenetic alopecia'
 'alopecia areata' 'IGA glomerulonephritis' 'pathological myopia'
 'Graves disease' 'sclerosing cholangitis' 'gout' 'suntan'
 'ventricular fibrillation' 'osteoarthritis, knee']

BIOLOGICAL PROCESS — 0 unique entries
[]

MEASUREMENT — 8178 unique entries
['temporal measurement' 'anthropometric measurement' 'erythrocyte count'
 'glucose tolerance test' 'vital capacity' 'forced expiratory volume'
 'body weights and measures' 'electrocardiography' 'body weight'
 'waist-hip ratio' 'neuroimaging measurement' 'hematocrit'
 'C-reactive protein measure